<a href="https://colab.research.google.com/github/RohitKhobare/ML-Lab-Assignments/blob/main/ML_LA1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import os
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. Download Uber Fares dataset directly via KaggleHub
print("Downloading Uber Fares dataset from Kaggle...")
path = kagglehub.dataset_download("yasserh/uber-fares-dataset")

# Locate the downloaded CSV file
csv_file = [os.path.join(path, f) for f in os.listdir(path) if f.endswith('.csv')][0]
df = pd.read_csv(csv_file)
print(f"Dataset loaded successfully! Initial shape: {df.shape}")

# 2. Data Pre-processing & Feature Engineering
cols_to_drop = [col for col in ['Unnamed: 0', 'key'] if col in df.columns]
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)

df.dropna(inplace=True)
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])

def haversine_np(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return 6371 * c

df['distance_km'] = haversine_np(
    df['pickup_longitude'].values,
    df['pickup_latitude'].values,
    df['dropoff_longitude'].values,
    df['dropoff_latitude'].values
)

df['hour'] = df['pickup_datetime'].dt.hour
df['day'] = df['pickup_datetime'].dt.day
df['month'] = df['pickup_datetime'].dt.month
df['dayofweek'] = df['pickup_datetime'].dt.dayofweek

# 3. Outlier Removal
df = df[(df['fare_amount'] >= 2.5) & (df['fare_amount'] <= 100)]
df = df[(df['distance_km'] > 0.05) & (df['distance_km'] <= 80)]
df = df[(df['passenger_count'] > 0) & (df['passenger_count'] <= 6)]

print(f"Shape after outlier removal: {df.shape}")

# 4. Train-Test Split
X = df[['distance_km', 'passenger_count', 'hour', 'day', 'month', 'dayofweek']]
y = df['fare_amount']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Model Training & Prediction
print("\nTraining Linear Regression...")
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

print("Training Random Forest Regressor...")
rf_model = RandomForestRegressor(n_estimators=50, max_depth=12, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

# 6. Model Evaluation Metrics
def calculate_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return mse, rmse, mae, r2

mse_lr, rmse_lr, mae_lr, r2_lr = calculate_metrics(y_test, y_pred_lr)
mse_rf, rmse_rf, mae_rf, r2_rf = calculate_metrics(y_test, y_pred_rf)

print("\n" + "="*65)
print(f"{'Metric':<10} | {'Linear Regression':<20} | {'Random Forest':<20}")
print("="*65)
print(f"{'MSE':<10} | {mse_lr:<20.4f} | {mse_rf:<20.4f}")
print(f"{'RMSE':<10} | {rmse_lr:<20.4f} | {rmse_rf:<20.4f}")
print(f"{'MAE':<10} | {mae_lr:<20.4f} | {mae_rf:<20.4f}")
print(f"{'R2 Score':<10} | {r2_lr:<20.4f} | {r2_rf:<20.4f}")
print("="*65)

Using Colab cache for faster access to the 'uber-fares-dataset' dataset.
Dataset loaded successfully! Initial shape: (200000, 9)
Shape after outlier removal: (192246, 12)

Training Linear Regression...
Training Random Forest Regressor...

Metric     | Linear Regression    | Random Forest       
MSE        | 16.8375              | 15.7282             
RMSE       | 4.1034               | 3.9659              
MAE        | 2.2523               | 2.1775              
R2 Score   | 0.8032               | 0.8162              
